# Hybrid Beam Forming using a Multilayer Perceptron 

## Load Helpers 

In [49]:
import numpy as np
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader, random_split 

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "scripts")); 
sys.path.append(str(Path.cwd().parent / "outputs")); 
from dataset_handler import DatasetHandler 
from mlp import MLP 

from sionna.rt import PathSolver
from generate_raynet_dataset_v3 import (
    CONFIG,
    setup_scene,
    make_tx_positions,
    make_candidate_configs,
    sample_two_users_for_target_class,
    trace_four_links,
    score_all_joint_configs,
    target_class_passes,
    thermal_noise_watts,
    dbm_to_watts,
) 

MODEL_CONFIG = {
    "input": 8, 
    "h1_dim": 128, 
    "h2_dim": 128, 
    "h3_dim": 64, 
    "out_1": len(CONFIG["sector_angles_deg"]),
    "out_2": CONFIG["num_codebooks"]
}; 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); 

## Train MLP 

In [ ]:
def train(trainset): 
    BATCH_SIZE = 10; 
    EPOCHS = 100; 
    LR = 1e-3; 

    dataset = DatasetHandler(trainset); 

    train_size = int(0.8 * len(dataset)); 
    val_size = len(dataset) - train_size; 

    train_set, val_set = random_split(
        dataset,
        [train_size, val_size]
    ); 

    train_loader = DataLoader(
        train_set,
        batch_size = BATCH_SIZE,
        shuffle = True
    ); 

    val_loader = DataLoader(
        val_set,
        batch_size = BATCH_SIZE,
        shuffle = False
    ); 


    model = MLP(
        input = MODEL_CONFIG["input"],
        h1_dim = MODEL_CONFIG["h1_dim"],
        h2_dim = MODEL_CONFIG["h2_dim"],
        h3_dim = MODEL_CONFIG["h3_dim"],
        out_1 = MODEL_CONFIG["out_1"],
        out_2 = MODEL_CONFIG["out_2"],
    ); 

    criterion = nn.CrossEntropyLoss(); 

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr = LR
    ); 


    for epoch in range(EPOCHS):

        model.train(); 

        running_loss = 0.0; 

        for (
            x,
            tx0_sector,
            tx1_sector,
            tx0_codebook,
            tx1_codebook
        ) in train_loader:

            x = x.to(DEVICE); 

            tx0_sector = tx0_sector.to(DEVICE); 
            tx1_sector = tx1_sector.to(DEVICE); 

            tx0_codebook = tx0_codebook.to(DEVICE); 
            tx1_codebook = tx1_codebook.to(DEVICE); 

            optimizer.zero_grad(); 

            s0, s1, c0, c1 = model(x); 

            loss_s0 = criterion(s0, tx0_sector); 
            loss_s1 = criterion(s1, tx1_sector); 

            loss_c0 = criterion(c0, tx0_codebook); 
            loss_c1 = criterion(c1, tx1_codebook); 

            loss = (
                loss_s0
                + loss_s1
                + loss_c0
                + loss_c1
            ); 

            loss.backward(); 
            optimizer.step(); 

            running_loss += loss.item(); 

        avg_train_loss = running_loss / len(train_loader); 

        print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Loss: {avg_train_loss:.4f}"
        ); 

    torch.save(
        model.state_dict(),
        "../outputs/beam_classifier.pth" 
    ); 

    print("Model saved."); 

train("../Datasets/dataset_8sector_3book_full_50.csv"); 

Epoch [1/100] Loss: 3.3309
Epoch [2/100] Loss: 2.8385
Epoch [3/100] Loss: 2.7081
Epoch [4/100] Loss: 2.6255
Epoch [5/100] Loss: 2.5519
Epoch [6/100] Loss: 2.4169
Epoch [7/100] Loss: 2.2243
Epoch [8/100] Loss: 2.0896
Epoch [9/100] Loss: 2.0015
Epoch [10/100] Loss: 1.9037
Epoch [11/100] Loss: 1.8497
Epoch [12/100] Loss: 1.8018
Epoch [13/100] Loss: 1.7712
Epoch [14/100] Loss: 1.7312
Epoch [15/100] Loss: 1.7073
Epoch [16/100] Loss: 1.6732
Epoch [17/100] Loss: 1.6523
Epoch [18/100] Loss: 1.6138
Epoch [19/100] Loss: 1.5582
Epoch [20/100] Loss: 1.5293
Epoch [21/100] Loss: 1.4916
Epoch [22/100] Loss: 1.4665
Epoch [23/100] Loss: 1.4319
Epoch [24/100] Loss: 1.4149
Epoch [25/100] Loss: 1.3930
Epoch [26/100] Loss: 1.3755
Epoch [27/100] Loss: 1.3577
Epoch [28/100] Loss: 1.3299
Epoch [29/100] Loss: 1.3250
Epoch [30/100] Loss: 1.3039
Epoch [31/100] Loss: 1.2840
Epoch [32/100] Loss: 1.2754
Epoch [33/100] Loss: 1.2689
Epoch [34/100] Loss: 1.2422
Epoch [35/100] Loss: 1.2437
Epoch [36/100] Loss: 1.2191
E

## Test MLP on Random User Pairs 
Generate two users in an environment and check if our MLP can determine the proper configuration to give the users the required throughput and maintain a sufficient signal to noise ratio. 

In [52]:
def test_single_pair(model, scene, solver, candidates, verbose = False): 
    tx0_pos, tx1_pos = make_tx_positions(); 
    rng = np.random.default_rng(); 

    # Generate two random users 
    tx0_target = rng.choice(CONFIG["sector_angles_deg"]); 
    tx1_target = rng.choice(CONFIG["sector_angles_deg"]); 

    users = sample_two_users_for_target_class(
        rng,
        tx0_target_angle=tx0_target,
        tx1_target_angle=tx1_target,
    ); 

    if users is None:
        return None; 

    users["u1_required_rate_bpshz"] = rng.uniform(1, 5); 
    users["u2_required_rate_bpshz"] = rng.uniform(1, 5); 

    links = trace_four_links(
        scene = scene,
        p_solver = solver,
        tx0_pos = tx0_pos,
        tx1_pos = tx1_pos,
        users = users,
        sample_seed = 12345,
    ); 


    # Build feature vector 
    noise = thermal_noise_watts(); 

    p_tx0 = dbm_to_watts(CONFIG["tx0_power_dbm"]); 
    p_tx1 = dbm_to_watts(CONFIG["tx1_power_dbm"]); 

    pilot_snr_u1 = max(
        p_tx0 * links["h11"]["power_linear"],
        p_tx1 * links["h21"]["power_linear"],
    ) / noise; 

    pilot_snr_u2 = max(
        p_tx0 * links["h12"]["power_linear"],
        p_tx1 * links["h22"]["power_linear"],
    ) / noise; 

    u1_pilot_snr_db = 10 * np.log10(pilot_snr_u1 + 1e-30); 
    u2_pilot_snr_db = 10 * np.log10(pilot_snr_u2 + 1e-30); 

    x = np.array([
        users["u1_distance_m"],
        users["u1_angle_deg"],
        users["u1_required_rate_bpshz"],
        u1_pilot_snr_db,

        users["u2_distance_m"],
        users["u2_angle_deg"],
        users["u2_required_rate_bpshz"],
        u2_pilot_snr_db,
    ], dtype=np.float32); 

    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(DEVICE); 


    # Use our MLP to determine the best code sector and beam indices ! 
    with torch.no_grad():
        logits_s0, logits_s1, logits_c0, logits_c1 = model(x); 

        tx0_sector = logits_s0.argmax(dim=1).item(); 
        tx1_sector = logits_s1.argmax(dim=1).item(); 

        tx0_codebook = logits_c0.argmax(dim=1).item(); 
        tx1_codebook = logits_c1.argmax(dim=1).item(); 

    if verbose: 
        print(); 
        print("Predicted Configuration"); 
        print("----------------------"); 
        print(f"TX0 sector   = {tx0_sector}"); 
        print(f"TX0 codebook = {tx0_codebook}"); 
        print(f"TX1 sector   = {tx1_sector}"); 
        print(f"TX1 codebook = {tx1_codebook}"); 

    pred_tx0_idx = candidates[(candidates["sector_idx"] == tx0_sector) & (candidates["codebook_idx"] == tx0_codebook)].index[0]; 
    pred_tx1_idx = candidates[(candidates["sector_idx"] == tx1_sector) & (candidates["codebook_idx"] == tx1_codebook)].index[0]; 


    # Evaluate predicted configuration 
    score_data = score_all_joint_configs(
        users = users,
        links = links,
        candidates = candidates,
    ); 

    passes, metrics = target_class_passes(
        score_data = score_data,
        users = users,
        target_tx0_idx = pred_tx0_idx,
        target_tx1_idx = pred_tx1_idx,
    ); 

    # Check optimality 
    best_tx0_idx = score_data["best_tx0_idx"]; 
    best_tx1_idx = score_data["best_tx1_idx"]; 

    best_tx0 = candidates.iloc[best_tx0_idx]; 
    best_tx1 = candidates.iloc[best_tx1_idx]; 

    if verbose: 
        print(); 
        print("Optimal Configuration"); 
        print("----------------------"); 
        print(
            f"TX0: sector = {best_tx0['sector_idx']} "
            f"codebook = {best_tx0['codebook_idx']}"
        ); 

        print(
            f"TX1: sector = {best_tx1['sector_idx']} "
            f"codebook = {best_tx1['codebook_idx']}"
        ); 

        print(); 
        print("Metrics"); 
        print("----------------------"); 

        for k, v in metrics.items():
            print(f"{k}: {v}"); 

        print(); 
        print("Prediction passes thresholds:", passes); 

    return passes; 


# Load model 
model = MLP(
    input = MODEL_CONFIG["input"],
    h1_dim = MODEL_CONFIG["h1_dim"],
    h2_dim = MODEL_CONFIG["h2_dim"],
    h3_dim = MODEL_CONFIG["h3_dim"],
    out_1 = MODEL_CONFIG["out_1"],
    out_2 = MODEL_CONFIG["out_2"],
); 

model.load_state_dict(torch.load("../outputs/beam_classifier.pth", map_location=DEVICE)); 

model.eval(); 
model.to(DEVICE); 

# Setup environment 
scene = setup_scene(); 
solver = PathSolver(); 
candidates = make_candidate_configs(); 

test_single_pair(model, scene, solver, candidates, verbose = True); 


Predicted Configuration
----------------------
TX0 sector   = 7
TX0 codebook = 1
TX1 sector   = 1
TX1 codebook = 1

Optimal Configuration
----------------------
TX0: sector = 7.0 codebook = 1.0
TX1: sector = 1.0 codebook = 1.0

Metrics
----------------------
target_score: -37.52070770886426
best_score: -37.52070770886426
target_is_best: 1
target_sinr_u1_db: -6.751983011183734
target_sinr_u2_db: 3.934765516574254
target_rate_u1_bpshz: 0.27649955162763934
target_rate_u2_bpshz: 1.7967795779640523
target_sum_rate_bpshz: 2.0732791295916915

Prediction passes thresholds: True


In [ ]:
# Test our MLP on N user pairs ! 
N = 100; 

success = 0; 
gen_err = 0; 
for _ in range(N): 
    result = test_single_pair(model, scene, solver, candidates); 
 
    if result is None: # Failed to generate a sample, does not count towards incorrect classifications 
        gen_err += 1; 

    elif result: 
        success += 1; 

print(f"Pass Rate: { success / (N - gen_err) * 100 }%"); 

KeyboardInterrupt: 